# 05 · Task 2 reasoning-trace analysis (offline)

Joins the Qwen3-32B raw log with its evaluated `per_example.csv`, extracts trace features, and runs the statistical tests.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "common").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from common import config
# Optional override of the project root for outputs/results (used for offline fixture tests).
if os.environ.get("A1_ROOT_OVERRIDE"):
    config.ROOT_DIR = Path(os.environ["A1_ROOT_OVERRIDE"])

LIMIT = None  # set to an int for a smoke run; falls back to A1_LIMIT
LIMIT = config.resolve_limit(LIMIT)
OUT = config.out_root(LIMIT)
RES = config.results_root(LIMIT)
print("LIMIT =", LIMIT, "| outputs:", OUT, "| results:", RES)

LIMIT = None | outputs: /home/nguyen/School/CISC_451/assignment1/outputs | results: /home/nguyen/School/CISC_451/assignment1/results


In [2]:
from common import report, traces, evaluate, data
from common.tokenizer import tokenizer_info
RUN = "qwen3_32b_thinking"
run_dir = OUT / RUN
pe = pd.read_csv(run_dir / "per_example.csv")
recs = {}
with open(run_dir / "raw.jsonl", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            r = json.loads(line)
        except json.JSONDecodeError:
            continue
        if r.get("phase", "full") != "full":
            continue
        # keep the last successful record per example (else the last one)
        eid = int(r["example_id"])
        if eid not in recs or r.get("status", "ok") == "ok":
            recs[eid] = r
print(len(pe), "evaluated rows;", len(recs), "raw records")

1034 evaluated rows; 1034 raw records


## Gold schema items and features

In [3]:
examples = {ex.example_id: ex for ex in data.load_examples(LIMIT)}
api = evaluate.import_official()
tables_json = config.DATA_DIR / "tables.json"
feats = []
with evaluate.official_cwd():
    for eid in pe["example_id"]:
        ex = examples[int(eid)]
        try:
            items = traces.gold_schema_items(ex.gold_sql, ex.db_id, api, tables_json)
        except Exception as e:  # parser failure: no coverage
            print(f"gold parse failed for {eid}: {e}")
            items = (set(), set())
        rec = recs.get(int(eid), {"reasoning_missing": True})
        feats.append(traces.extract_features(rec, ex, items).to_dict())
fdf = pd.DataFrame(feats).merge(pe[["example_id", "difficulty", "ex"]], on="example_id")
excl = (fdf[fdf.excluded].groupby("reason").size().rename("count").reset_index()
        if fdf.excluded.any() else pd.DataFrame({"reason": [], "count": []}))
excl = pd.concat([excl, pd.DataFrame([{"reason": "included", "count": int((~fdf.excluded).sum())}])])
report.export_table(excl, "task2_exclusions", "Examples excluded from trace analysis, by reason.", RES)
df = fdf[~fdf.excluded].copy()
df["correct"] = df["ex"].map({1: "correct", 0: "incorrect"})
FEATS = list(traces.FEATURES)
display(excl)

# Length validation: tokenizer count of the trace text vs provider-reported numbers.
comp = pd.Series({int(r["example_id"]): r.get("completion_tokens") for r in recs.values()})
v = df.assign(completion_tokens=df["example_id"].map(comp))
val = pd.DataFrame([{
    "n": len(v),
    "mean_text_tokens": v["length_tokens"].mean(),
    "mean_provider_reasoning_tokens": v["provider_reasoning_tokens"].mean(),
    "mean_provider_completion_tokens": v["completion_tokens"].mean(),
    "mean_chars": v["length_chars"].mean(),
    "provider_reasoning_gt_completion": int((v["provider_reasoning_tokens"] > v["completion_tokens"]).sum()),
    "text_tokens_gt_completion": int((v["length_tokens"] > v["completion_tokens"]).sum()),
    "spearman_text_vs_provider": v["length_tokens"].corr(v["provider_reasoning_tokens"], method="spearman"),
}])
info = tokenizer_info()
report.export_table(val, "task2_length_validation",
    f"Trace length measured by tokenizing the reasoning text ({info['tokenizer']}, tokenizers "
    f"{info['tokenizers_version']}) versus provider-reported counts. Provider "
    "reasoning_tokens is an estimate (DeepInfra) and can exceed completion_tokens, so length_tokens "
    "(text) is used in all tests; length_chars is reported as a robustness check.", RES)
display(val)

,reason,count
0,included,1034.0


,n,mean_text_tokens,mean_provider_reasoning_tokens,mean_provider_completion_tokens,mean_chars,provider_reasoning_gt_completion,text_tokens_gt_completion,spearman_text_vs_provider
0,1034,937.073501,1023.610251,981.472921,4041.674081,775,0,0.994361


## Descriptive statistics

In [4]:
d_corr = traces.describe(df, FEATS, "correct")
d_diff = traces.describe(df, FEATS, "difficulty")
report.export_table(d_corr, "task2_desc_by_correctness", "Trace features by EX correctness "
                    "(count, mean, median, quartiles).", RES)
report.export_table(d_diff, "task2_desc_by_difficulty", "Trace features by difficulty "
                    "(count, mean, median, quartiles).", RES)
display(d_corr); display(d_diff)

,feature,correct,count,mean,median,q1,q3,iqr
0,length_tokens,correct,809,778.770087,517.000000,356.000000,885.000000,529.000000
1,length_tokens,incorrect,225,1506.262222,1189.000000,696.000000,1929.000000,1233.000000
2,length_chars,correct,809,3367.501854,2271.000000,1586.000000,3912.000000,2326.000000
3,length_chars,incorrect,225,6465.697778,5046.000000,3186.000000,8361.000000,5175.000000
4,marker_count,correct,809,4.959209,4.000000,2.000000,7.000000,5.000000
5,marker_count,incorrect,225,8.440000,8.000000,5.000000,11.000000,6.000000
6,markers_per_1k,correct,809,6.906834,6.622517,4.972875,8.658009,3.685133
7,markers_per_1k,incorrect,225,6.469311,6.247560,4.464286,7.887818,3.423532
8,sql_drafts,correct,809,2.907293,1.000000,1.000000,3.000000,2.000000
9,sql_drafts,incorrect,225,5.302222,3.000000,1.000000,5.000000,4.000000


,feature,difficulty,count,mean,median,q1,q3,iqr
0,length_tokens,easy,248,507.439516,362.000000,278.500000,554.500000,276.000000
1,length_tokens,extra,166,1594.186747,1267.500000,850.250000,1990.750000,1140.500000
2,length_tokens,hard,174,1396.183908,1073.000000,637.500000,1901.500000,1264.000000
3,length_tokens,medium,446,752.282511,503.500000,381.250000,804.750000,423.500000
4,length_chars,easy,248,2270.229839,1629.000000,1262.000000,2432.750000,1170.750000
5,length_chars,extra,166,6768.253012,5496.000000,3678.750000,8741.750000,5063.000000
6,length_chars,hard,174,5999.816092,4432.000000,2714.750000,8004.000000,5289.250000
7,length_chars,medium,446,3247.928251,2240.000000,1705.250000,3477.000000,1771.750000
8,marker_count,easy,248,3.584677,3.000000,2.000000,5.000000,3.000000
9,marker_count,extra,166,8.783133,8.000000,5.000000,12.000000,7.000000


## Tests (Holm-adjusted)

In [5]:
mwu = traces.mann_whitney_table(df, FEATS, "ex")
kw = traces.kruskal_table(df, FEATS, "difficulty")
sp = traces.spearman_table(df, FEATS, "difficulty")
report.export_table(mwu, "task2_mannwhitney", "Mann-Whitney U, correct vs incorrect, with rank-biserial "
                    "effect size; Holm-adjusted p-values across features.", RES)
report.export_table(kw, "task2_kruskal", "Kruskal-Wallis across difficulty levels; Holm-adjusted.", RES)
report.export_table(sp, "task2_spearman", "Spearman correlation with ordinal difficulty "
                    "(easy=1..extra=4); Holm-adjusted.", RES)
display(mwu); display(kw); display(sp)

,feature,n_correct,n_incorrect,n_dropped,U,rank_biserial,p_raw,p_holm
0,length_tokens,809,225,0,45442.5,-0.500700,1.306247e-30,7.837483e-30
1,length_chars,809,225,0,45057.0,-0.504936,4.212009e-31,2.948406e-30
2,marker_count,809,225,0,49568.0,-0.455372,7.143987e-26,3.571993e-25
3,markers_per_1k,809,225,0,100117.0,0.100036,2.158048e-02,4.316097e-02
4,sql_drafts,809,225,0,62934.5,-0.308507,3.365358e-14,1.346143e-13
5,table_coverage,809,225,0,91417.0,0.004444,5.824477e-02,5.824477e-02
6,column_coverage,769,223,42,90124.0,0.051088,5.061593e-05,1.518478e-04


,feature,n,n_groups,n_dropped,H,p_raw,p_holm
0,length_tokens,1034,4,0,331.570212,1.458730e-71,1.021111e-70
1,length_chars,1034,4,0,323.683170,7.437524e-70,4.462514e-69
2,marker_count,1034,4,0,220.032949,1.975134e-47,7.900535e-47
3,markers_per_1k,1034,4,0,32.245319,4.646037e-07,1.393811e-06
4,sql_drafts,1034,4,0,236.834167,4.603610e-51,2.301805e-50
5,table_coverage,1034,4,0,5.228916,1.557820e-01,1.557820e-01
6,column_coverage,992,4,42,18.347864,3.728442e-04,7.456884e-04


,feature,n,n_dropped,rho,p_raw,p_holm
0,length_tokens,1034,0,0.559686,2.720914e-86,1.904640e-85
1,length_chars,1034,0,0.552236,1.341715e-83,8.050291e-83
2,marker_count,1034,0,0.454562,7.351636e-54,3.675818e-53
3,markers_per_1k,1034,0,-0.155709,4.863229e-07,1.458969e-06
4,sql_drafts,1034,0,0.440496,2.559018e-50,1.023607e-49
5,table_coverage,1034,0,-0.047764,1.248051e-01,1.248051e-01
6,column_coverage,992,42,-0.087695,5.711616e-03,1.142323e-02


## Logistic regression

In [6]:
lr = traces.fit_logit(df)
report.export_table(lr.table, "task2_logit", f"{lr.label}: EX on z-scored trace features plus "
                    f"difficulty (reference: easy), n={lr.n}. table_coverage is omitted (1.0 for all but "
                    "one trace, causing quasi-separation); length_chars is omitted (collinear "
                    "with length_tokens).", RES)
print(lr.label, "| converged:", lr.converged)
lr.table

Logistic regression (maximum likelihood) | converged: True


,term,coef,se,z,p,odds_ratio,or_ci_low,or_ci_high,n
0,Intercept,1.951555,0.234604,8.318500,8.908353e-17,7.039627,4.444820,11.149237,992
1,"C(difficulty, Treatment('easy'))[T.medium]",-0.420867,0.267767,-1.571767,1.160046e-01,0.656477,0.388415,1.109541,992
2,"C(difficulty, Treatment('easy'))[T.hard]",-0.792116,0.305125,-2.596034,9.430661e-03,0.452885,0.249038,0.823591,992
3,"C(difficulty, Treatment('easy'))[T.extra]",-1.250274,0.299198,-4.178746,2.931209e-05,0.286426,0.159344,0.514862,992
4,z_length_tokens,-0.927260,0.129216,-7.176031,7.176452e-13,0.395636,0.307119,0.509666,992
5,z_markers_per_1k,-0.153362,0.093226,-1.645059,9.995756e-02,0.857819,0.714565,1.029792,992
6,z_sql_drafts,0.408750,0.118641,3.445262,5.705068e-04,1.504935,1.192697,1.898914,992
7,z_column_coverage,0.326578,0.084626,3.859090,1.138099e-04,1.386217,1.174350,1.636307,992


## Box plots

In [7]:
def boxes(by, order, name, title):
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    for ax in axes.flat[len(FEATS):]:
        ax.axis("off")
    for ax, f in zip(axes.flat, FEATS):
        groups = [pd.to_numeric(df.loc[df[by] == g, f], errors="coerce").dropna() for g in order]
        ax.boxplot(groups)
        ax.set_xticks(range(1, len(order) + 1), order)
        ax.set_title(f, fontsize=9)
    fig.suptitle(title); fig.tight_layout()
    print(report.save_figure(fig, name, RES)); plt.close(fig)

boxes("correct", ["correct", "incorrect"], "task2_box_by_correctness", "Trace features by correctness")
boxes("difficulty", ["easy", "medium", "hard", "extra"], "task2_box_by_difficulty",
      "Trace features by difficulty")

{'png': PosixPath('/home/nguyen/School/CISC_451/assignment1/results/task2_box_by_correctness.png'), 'pdf': PosixPath('/home/nguyen/School/CISC_451/assignment1/results/task2_box_by_correctness.pdf')}


{'png': PosixPath('/home/nguyen/School/CISC_451/assignment1/results/task2_box_by_difficulty.png'), 'pdf': PosixPath('/home/nguyen/School/CISC_451/assignment1/results/task2_box_by_difficulty.pdf')}
